# Baseline Model: DenseNet-121

## Import Dependencies

In [1]:
# Pip installations if necessary 
# %pip install torch torchvision torchaudio

In [10]:
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
import glob

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

## CONFIGURATION & HYPERPARAMETERS

In [ ]:
Config = {
    "IMG_SIZE": 224,           # Standard for DenseNet 
    "BATCH_SIZE": 16,          # Lower this if you run out of memory
    "LEARNING_RATE": 1e-4,     # Standard starting rate for fine-tuning
    "EPOCHS": 10,
    "DATA_DIR": "images_001/images",      # Directory said to subset of images due to memory constraints
    "CSV_FILE": "Data_Entry_2017.csv",
    "SPLIT_FILE": "data/train_val_list.txt"
}

# Device config (MPS for Mac, CUDA for NVIDIA, CPU otherwise)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: mps


## CUSTOM TRANSFORMS
### Please reference data_loader.ipynb for more detail

In [ ]:
class LetterboxPad:
    """Resizes image to fit within target size while keeping aspect ratio, padding the rest."""
    def __call__(self, image):
        w, h = image.size
        target_h, target_w = Config["IMG_SIZE"], Config["IMG_SIZE"]
        
        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)
        image = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
        
        new_img = Image.new("RGB", (target_w, target_h), (0, 0, 0))
        new_img.paste(image, ((target_w - new_w) // 2, (target_h - new_h) // 2))
        return new_img

## Dataset Class

In [ ]:
class NIHDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None, classes=None):
        self.df = dataframe
        self.root_dir = root_dir
        self.transform = transform
        self.classes = classes
        
        # Pre-process labels: "No Finding" -> Empty List
        self.df['labels_list'] = self.df['Finding Labels'].apply(
            lambda x: [] if 'No Finding' in x else x.split('|')
        )
        
        # Binarize labels
        if self.classes is None:
            self.mlb = MultiLabelBinarizer()
            self.labels = self.mlb.fit_transform(self.df['labels_list'])
            self.classes = self.mlb.classes_
        else:
            self.mlb = MultiLabelBinarizer(classes=self.classes)
            self.labels = self.mlb.fit_transform(self.df['labels_list'])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['Image Index']
        # Handle possible subfolders if using the extracted NIH structure
        # For now, assuming all images are in root_dir or we search for them
        img_path = os.path.join(self.root_dir, img_name)
        
        # Read with OpenCV (to apply CLAHE)
        image = cv2.imread(img_path)
        if image is None:
            # Fallback for missing files (safety check)
            print(f"Warning: Could not load {img_path}")
            image = np.zeros((224, 224, 3), dtype=np.uint8)
        
        # Convert to Grayscale for CLAHE
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        
        # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        image = clahe.apply(gray)
        
        # Convert back to RGB for the model (DenseNet expects 3 channels)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        # Convert to PIL for Torchvision Transforms
        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return image, label

## Preparation & Loading

In [ ]:
# Load Metadata
full_df = pd.read_csv(Config["CSV_FILE"])

# Load Train/Val List
with open(Config["SPLIT_FILE"], 'r') as f:
    train_val_list = [line.strip() for line in f.readlines()]

# Filter DataFrame to include only images in the official Train/Val split
train_val_df = full_df[full_df['Image Index'].isin(train_val_list)]

train_val_df['file_exists'] = train_val_df['Image Index'].apply(
    lambda x: os.path.exists(os.path.join(Config["DATA_DIR"], x))
)
print(f"Original Count: {len(train_val_df)}")
train_val_df = train_val_df[train_val_df['file_exists']].copy()
print(f"Available for Training: {len(train_val_df)}")

# Custom Train/Val Split by PATIENT ID (Prevent Data Leakage)
patient_ids = train_val_df['Patient ID'].unique()
train_patients, val_patients = train_test_split(patient_ids, test_size=0.2, random_state=42)

train_df = train_val_df[train_val_df['Patient ID'].isin(train_patients)]
val_df = train_val_df[train_val_df['Patient ID'].isin(val_patients)]

print(f"Training on {len(train_df)} images.")
print(f"Validating on {len(val_df)} images.")

# Define Transforms
train_transforms = transforms.Compose([
    LetterboxPad(),
    transforms.RandomHorizontalFlip(), # Augmentation
    transforms.RandomRotation(10),     # Augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    LetterboxPad(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create Datasets
# Define classes alphabetically
nih_classes = sorted(['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 
                      'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 
                      'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'])

train_dataset = NIHDataset(train_df, Config["DATA_DIR"], transform=train_transforms, classes=nih_classes)
val_dataset = NIHDataset(val_df, Config["DATA_DIR"], transform=val_transforms, classes=nih_classes)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=Config["BATCH_SIZE"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=Config["BATCH_SIZE"], shuffle=False, num_workers=0)

Original Count: 86524
Available for Training: 4032
Training on 3183 images.
Validating on 849 images.


## Model Definition

In [ ]:
def get_densenet_model(num_classes):
    # Load Pretrained DenseNet121
    model = models.densenet121(weights='DEFAULT')
    
    # Replace Classifier
    # DenseNet's classifier is a Linear layer: (classifier): Linear(in_features=1024, out_features=1000, bias=True)
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_features, num_classes),
    )
    return model

model = get_densenet_model(len(nih_classes))
model = model.to(device)

## Loss & Optimizer

In [8]:
# Calculate Pos_Weights for Imbalance
# Weight = (Total - Positive) / Positive
class_counts = train_dataset.labels.sum(axis=0)
pos_weights = (len(train_df) - class_counts) / (class_counts + 1e-5) # Avoid div by zero
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=Config["LEARNING_RATE"])

## Training Loop

In [ ]:
best_val_loss = float('inf')

print("\nStarting Training...")
for epoch in range(Config["EPOCHS"]):
    model.train()
    running_loss = 0.0
    
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        if i % 100 == 0:
            print(f"Epoch [{epoch+1}/{Config['EPOCHS']}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")
            
    # Validation Phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in enumerate(val_loader):
            # Fixing logic for clean paste:
            pass 
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch [{epoch+1}] Complete. Avg Train Loss: {running_loss/len(train_loader):.4f}, Avg Val Loss: {avg_val_loss:.4f}")
    
    # Save Best Model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "best_densenet_model.pth")
        print("Model Saved!")

print("Training Complete.")


Starting Training...
Epoch [1/10], Step [0/199], Loss: 1.5271
Epoch [1/10], Step [100/199], Loss: 1.0939


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Epoch [1] Complete. Avg Train Loss: 1.2920, Avg Val Loss: 1.2452
Model Saved!
Epoch [2/10], Step [0/199], Loss: 0.8859
Epoch [2/10], Step [100/199], Loss: 1.0003


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Epoch [2] Complete. Avg Train Loss: 1.0988, Avg Val Loss: 1.2075
Model Saved!
Epoch [3/10], Step [0/199], Loss: 1.1411


...

Epoch [8] Complete. Avg Train Loss: 0.6651, Avg Val Loss: 1.6856
Epoch [9/10], Step [0/199], Loss: 0.6230
Epoch [9/10], Step [100/199], Loss: 0.5563


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Epoch [9] Complete. Avg Train Loss: 0.5590, Avg Val Loss: 2.0174
Epoch [10/10], Step [0/199], Loss: 0.3196
Epoch [10/10], Step [100/199], Loss: 0.7301


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


Epoch [10] Complete. Avg Train Loss: 0.5325, Avg Val Loss: 2.2275
Training Complete.
